In [12]:
import sys
from pathlib import Path

root = Path().resolve()
sys.path.insert(0, str(root / "src"))


In [13]:
import tensorflow as tf

from FLRW_Net.network.network import NeuralNetwork

tf.keras.backend.set_floatx("float64")

In [ ]:
flrw_net = NeuralNetwork(number_of_timesteps=1, triangulation="16-cell", cosmological_constant=1e-3)
adam_optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-3,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-7,
    amsgrad=False,
    decay=0.9,
    clipnorm=1,
    clipvalue=None,
    global_clipnorm=None,
)
flrw_net.compile(optimizer=adam_optimizer)

In [ ]:
import numpy as np
# inputs = tf.constant([[1, 2/5-3/8, 2, 2/5-3/8, 3]], dtype=tf.float64)  # 5-cell
inputs = tf.constant([[1, 1/2-3/8, 2]], dtype=tf.float64)  # 16-cell
# inputs = tf.constant([[1, (3 + np.sqrt(5)) / 2, 2]], dtype=tf.float64)  # 600-cell
flrw_net.training(inputs, epochs=10000)

In [ ]:
for key, value in flrw_net.model_params._asdict().items():
    print(key, value.numpy())
tf.print(flrw_net(inputs))

In [ ]:
from FLRW_Net.utils.losses import strut_losses, spatial_edge_losses

prediction = tf.constant([[1, 0.7071313364895393, 2]], dtype=tf.float64)
loss_struts = strut_losses(prediction, flrw_net.model_params)
loss_spatial_edges = spatial_edge_losses(prediction, flrw_net.model_params)
combined = tf.concat([loss_struts, loss_spatial_edges], axis=1)
loss = tf.squeeze(tf.reduce_mean(combined, axis=1))
tf.print(loss)

In [19]:
# Check whether the forward feed gives the same result
import tensorflow as tf

from FLRW_Net.network.network import NeuralNetwork

from src.OneStep.layer import HiddenLayer as Layer1
from src.TwoStep.layer import HiddenLayer as Layer2
from src.ThreeStep.layer import HiddenLayer as Layer3
from src.FourStep.layer import HiddenLayer as Layer4

layer1 = Layer1()
layer2 = Layer2()
layer3 = Layer3()
layer4 = Layer4()
flrw_net_1 = NeuralNetwork(number_of_timesteps=1, triangulation="5-cell", cosmological_constant=1e-3)
flrw_net_2 = NeuralNetwork(number_of_timesteps=2, triangulation="5-cell", cosmological_constant=1e-3)
flrw_net_3 = NeuralNetwork(number_of_timesteps=3, triangulation="5-cell", cosmological_constant=1e-3)
flrw_net_4 = NeuralNetwork(number_of_timesteps=4, triangulation="5-cell", cosmological_constant=1e-3)

inputs_1 = tf.constant([[1, 0.67, 2]], dtype=tf.float64)
inputs_2 = tf.constant([[1, 0.67, 2, 0.68, 3]], dtype=tf.float64)
inputs_3 = tf.constant([[1, 0.67, 2, 0.68, 3, 0.69, 4]], dtype=tf.float64)
inputs_4 = tf.constant([[1, 0.67, 2, 0.68, 3, 0.69, 4, 0.7, 5]], dtype=tf.float64)

print(layer1(inputs_1))
print(flrw_net_1(inputs_1))
print()
print(layer2(inputs_2))
print(flrw_net_2(inputs_2))
print()
print(layer3(inputs_3))
print(flrw_net_3(inputs_3))
print()
print(layer4(inputs_4))
print(flrw_net_4(inputs_4))

tf.Tensor([[1.         1.02225242 2.        ]], shape=(1, 3), dtype=float64)
tf.Tensor([[1.         1.02225242 2.        ]], shape=(1, 3), dtype=float64)

tf.Tensor([[1.         0.73019685 1.71430191 1.32058156 3.        ]], shape=(1, 5), dtype=float64)
tf.Tensor([[1.         0.73019685 1.71430191 1.32058156 3.        ]], shape=(1, 5), dtype=float64)

tf.Tensor(
[[1.         0.73019685 1.71430191 1.02021009 2.70756291 1.33378005
  4.        ]], shape=(1, 7), dtype=float64)
tf.Tensor(
[[1.         0.73019685 1.71430191 1.02021009 2.70756291 1.33378005
  4.        ]], shape=(1, 7), dtype=float64)

tf.Tensor(
[[1.         0.73019685 1.71430191 1.02021009 2.70756291 1.0251638
  3.70094989 1.34688383 5.        ]], shape=(1, 9), dtype=float64)
tf.Tensor(
[[1.         0.73019685 1.71430191 1.02021009 2.70756291 1.0251638
  3.70094989 1.34688383 5.        ]], shape=(1, 9), dtype=float64)
